In [1]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

Found existing installation: keras 3.10.0
Uninstalling keras-3.10.0:
  Successfully uninstalled keras-3.10.0
Found existing installation: matplotlib 3.10.0
Uninstalling matplotlib-3.10.0:
  Successfully uninstalled matplotlib-3.10.0
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
  Successfully uninstalled tensorflow-2.19.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip show openai_harmony

Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path

for file in Path('/kaggle/').rglob('*'):
    if file.is_file():
        print(file)

/kaggle/lib/kaggle/gcp.py
/kaggle/input/models/danielhanchen/gpt-oss-20b/transformers/default/1/model.safetensors.index.json
/kaggle/input/models/danielhanchen/gpt-oss-20b/transformers/default/1/model-00000-of-00002.safetensors
/kaggle/input/models/danielhanchen/gpt-oss-20b/transformers/default/1/USAGE_POLICY
/kaggle/input/models/danielhanchen/gpt-oss-20b/transformers/default/1/config.json
/kaggle/input/models/danielhanchen/gpt-oss-20b/transformers/default/1/model-00001-of-00002.safetensors
/kaggle/input/models/danielhanchen/gpt-oss-20b/transformers/default/1/LICENSE
/kaggle/input/models/danielhanchen/gpt-oss-20b/transformers/default/1/model-00002-of-00002.safetensors
/kaggle/input/models/danielhanchen/gpt-oss-20b/transformers/default/1/README.md
/kaggle/input/models/danielhanchen/gpt-oss-20b/transformers/default/1/tokenizer.json
/kaggle/input/models/danielhanchen/gpt-oss-20b/transformers/default/1/tokenizer_config.json
/kaggle/input/models/danielhanchen/gpt-oss-20b/transformers/defaul

In [4]:
import warnings
warnings.simplefilter('ignore')

In [5]:
import os
import sys
import subprocess

In [6]:
def set_env(input_archive, temp_dir):

    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir, exist_ok=True)
        
        subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)
    
    subprocess.run([
        sys.executable, 
        '-m', 
        'pip', 
        'install', 
        '--no-index', 
        '--find-links', 
        f'{temp_dir}/wheels', 
        'unsloth', 
        'trl', 
        'vllm', 
        'openai_harmony'
    ], check=True)

In [7]:
set_env(
    #input_archive='/kaggle/input/aimo-3-utils/wheels.tar.gz', 
    input_archive='/kaggle/input/notebooks/andreasbis/aimo-3-utils/wheels.tar.gz',
    temp_dir='/kaggle/tmp/setup'
)

Looking in links: /kaggle/tmp/setup/wheels
Processing /kaggle/tmp/setup/wheels/unsloth-2025.12.9-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/trl-0.24.0-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/vllm-0.11.2-cp38-abi3-manylinux1_x86_64.whl
Processing /kaggle/tmp/setup/wheels/openai_harmony-0.0.8-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing /kaggle/tmp/setup/wheels/unsloth_zoo-2025.12.7-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/tyro-1.0.3-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/xformers-0.0.33.post1-cp39-abi3-manylinux_2_28_x86_64.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/bitsandbytes-0.49.0-py3-none-manylinux_2_24_x86_64.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/datasets-4.3.0-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/transformers-4.57.3-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/prometheus_fastapi_instrumentator-7.1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyldavis 3.4.1 requires scikit-learn>=1.0.0, which is not installed.
ydata-profiling 4.18.1 requires matplotlib<=3.10,>=3.5, which is not installed.
librosa 0.11.0 requires scikit-learn>=1.1.0, which is not installed.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
bigframes 2.31.0 requires matplotlib>=3.7.1, which is not installed.
shap 0.50.0 requires scikit-learn, which is not installed.
pynndescent 0.6.0 requires scikit-learn>=0.18, which is not installed.
umap-learn 0.5.11 requires scikit-learn>=1.6, which is not installed.
sentence-transformers 5.2.0 requires scikit-learn, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
arviz 0.22.0 requires matplotlib>=3.8, which is not installed.
cuml-cu12 25.10

In [8]:
subprocess.run(['ls', '/kaggle/tmp/setup/tiktoken_encodings'])

cl100k_base.tiktoken
o200k_base.tiktoken


CompletedProcess(args=['ls', '/kaggle/tmp/setup/tiktoken_encodings'], returncode=0)

In [9]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/tmp/setup/tiktoken_encodings'

In [10]:
import gc
import re
import math
import time
import queue
import threading
import contextlib
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import pandas as pd
import polars as pl

from openai import OpenAI

from openai_harmony import (
    HarmonyEncodingName, 
    load_harmony_encoding, 
    SystemContent, 
    ReasoningEffort, 
    ToolNamespaceConfig, 
    Author, 
    Message, 
    Role, 
    TextContent, 
    Conversation
)

from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server

In [11]:
class CFG:
    
    system_prompt = (
        'You are an elite mathematical problem solver with expertise at the International '
        'Mathematical Olympiad (IMO) level. Your goal is to find the correct answer through '
        'rigorous mathematical reasoning.\n\n'
        
        '# Problem-Solving Approach:\n'
        '1. UNDERSTAND: Carefully read and rephrase the problem in your own words. '
        'Identify what is given, what needs to be found, and any constraints.\n'
        '2. EXPLORE: Consider multiple solution strategies. Think about relevant theorems, '
        'techniques, patterns, or analogous problems. Don\'t commit to one approach immediately.\n'
        '3. PLAN: Select the most promising approach and outline key steps before executing.\n'
        '4. EXECUTE: Work through your solution methodically. Show all reasoning steps clearly.\n'
        '5. VERIFY: Check your answer by substituting back, testing edge cases, or using '
        'alternative methods. Ensure logical consistency throughout.\n\n'
        
        '# Mathematical Reasoning Principles:\n'
        '- Break complex problems into smaller, manageable sub-problems\n'
        '- Look for patterns, symmetries, and special cases that provide insight\n'
        '- Use concrete examples to build intuition before generalizing\n'
        '- Consider extreme cases and boundary conditions\n'
        '- If stuck, try working backwards from the desired result\n'
        '- Be willing to restart with a different approach if needed\n\n'
        
        '# Verification Requirements:\n'
        '- Cross-check arithmetic and algebraic manipulations\n'
        '- Verify that your solution satisfies all problem constraints\n'
        '- Test your answer with simple cases or special values when possible\n'
        '- Ensure dimensional consistency and reasonableness of the result\n\n'

        '# Output Format:\n'
        'The final answer must be a non-negative integer between 0 and 99999.\n'
        'Place your final numerical answer inside \\boxed{}, e.g., \\boxed{42}\n\n'
        
        'Think step-by-step and show your complete reasoning process. Quality of reasoning '
        'is as important as the final answer.'
    )
    
    tool_prompt = (
        'Use this tool to execute Python code for:\n'
        '- Complex calculations that would be error-prone by hand\n'
        '- Numerical verification of analytical results\n'
        '- Generating examples or testing conjectures\n'
        '- Visualizing problem structure when helpful\n'
        '- Brute-force verification for small cases\n\n'
        
        'The environment is a stateful Jupyter notebook. Code persists between executions.\n'
        'Always use print() to display results. Write clear, well-commented code.\n\n'
        
        'Remember: Code should support your mathematical reasoning, not replace it. '
        'Explain what you\'re computing and why before running code.'
    )
    
    preference_prompt = (
        'You have access to `math`, `numpy`, and `sympy` for:\n\n'
        
        '# Symbolic Computation (sympy):\n'
        '- Algebraic manipulation and simplification\n'
        '- Solving equations and systems of equations\n'
        '- Symbolic differentiation and integration\n'
        '- Number theory functions (primes, divisors, modular arithmetic)\n'
        '- Polynomial operations and factorization\n'
        '- Working with mathematical expressions symbolically\n\n'
        
        '# Numerical Computation (numpy):\n'
        '- Array operations and linear algebra\n'
        '- Efficient numerical calculations for large datasets\n'
        '- Matrix operations and eigenvalue problems\n'
        '- Statistical computations\n\n'
        
        '# Mathematical Functions (math):\n'
        '- Standard mathematical functions (trig, log, exp)\n'
        '- Constants like pi and e\n'
        '- Basic operations for single values\n\n'
        
        'Best Practices:\n'
        '- Use sympy for exact symbolic answers when possible\n'
        '- Use numpy for numerical verification and large-scale computation\n'
        '- Combine symbolic and numerical approaches: derive symbolically, verify numerically\n'
        '- Document your computational strategy clearly\n'
        '- Validate computational results against known cases or theoretical bounds'
    )
    
    served_model_name = 'gpt-oss'
    #model_path = '/kaggle/input/gpt-oss-20b/transformers/default/1'
    model_path = '/kaggle/input/models/danielhanchen/gpt-oss-120b/transformers/default/1/'
    
    kv_cache_dtype = 'fp8_e4m3'
    dtype = 'auto'

    high_problem_timeout = 900
    base_problem_timeout = 300

    notebook_limit = 17400
    server_timeout = 180

    session_timeout = 960
    jupyter_timeout = 6
    sandbox_timeout = 3

    stream_interval = 200
    context_tokens = 65536
    buffer_tokens = 512
    search_tokens = 32
    top_logprobs = 5
    batch_size = 256
    early_stop = 5
    attempts = 5
    workers = 16
    turns = 128
    seed = 42

    gpu_memory_utilization = 0.96
    temperature = 0.5
    min_p = 0.02

In [12]:
def extract_answer(trace: str) -> Optional[int]:
    """Extract the final integer answer from a solution trace.
    Searches for \\boxed{N} first (competition standard), then
    falls back to the last bare integer in the trace.
    """
    # \\boxed{N} — last occurrence wins (model may self-correct)
    boxed = re.findall(r'\\\\boxed\{(\d+)\}', trace)
    if boxed:
        return int(boxed[-1])

    # 'answer is N' / 'answer = N' / 'answer: N'
    m = re.search(
        r'(?:answer|result)\s*(?:is|=|:)\s*(\d+)',
        trace, re.IGNORECASE
    )
    if m:
        return int(m.group(1))

    # Last integer ≤ 6 digits (avoids line-numbers / port numbers)
    ints = re.findall(r'\b(\d{1,6})\b', trace)
    if ints:
        return int(ints[-1])

    return None


In [13]:
set_seed(CFG.seed)

In [14]:
class AIMO3Template:

    def __init__(self):

        pass

    def get_system_content(self, system_prompt: str, tool_config: ToolNamespaceConfig) -> SystemContent:

        return (
            SystemContent.new()
            .with_model_identity(system_prompt)
            .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
            .with_tools(tool_config)
        )

    def apply_chat_template(
        self, 
        system_prompt: str, 
        user_prompt: str, 
        tool_config: ToolNamespaceConfig
    ) -> list[Message]:

        system_content = self.get_system_content(system_prompt, tool_config)        
        system_message = Message.from_role_and_content(Role.SYSTEM, system_content)

        user_message = Message.from_role_and_content(Role.USER, user_prompt)

        return [system_message, user_message]

In [15]:
class AIMO3Sandbox:

    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:

        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count

            return ports

    def __init__(self, timeout: float):

        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None
        
        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'
        env['PULP_CBC_CMD'] = 'cbc -silent'

        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        self.execute(
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
            'try:\n'
            '    import pulp; pulp.LpSolverDefault.msg = 0\n'
            'except Exception:\n'
            '    pass\n'
        )

    def _format_error(self, traceback: list[str]) -> str:

        clean_lines = []

        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)

            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue

            clean_lines.append(clean_frame)

        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:

        client = self._client
        effective_timeout = timeout or self._default_timeout
        
        msg_id = client.execute(
            code, 
            store_history=True, 
            allow_stdin=False, 
            stop_on_error=False
        )

        stdout_parts = []
        stderr_parts = []
        
        start_time = time.time()

        while True:
            elapsed = time.time() - start_time

            if elapsed > effective_timeout:
                self._km.interrupt_kernel()

                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0)

            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')

                if content.get('name') == 'stdout':
                    stdout_parts.append(text)

                else:
                    stderr_parts.append(text)

            elif msg_type == 'error':
                traceback_list = content.get('traceback', [])

                stderr_parts.append(self._format_error(traceback_list))

            elif msg_type in {'execute_result', 'display_data'}:
                data = content.get('data', {})
                text = data.get('text/plain')

                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

            elif msg_type == 'status':
                if content.get('execution_state') == 'idle':
                    break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr

        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    def close(self):

        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()

        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)

            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    def reset(self):
        
        self.execute(
            '%reset -f\n'
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
            'try:\n'
            '    import pulp; pulp.LpSolverDefault.msg = 0\n'
            'except Exception:\n'
            '    pass\n'

        )

    def __del__(self):

        self.close()

In [16]:
class AIMO3Tool:

    def __init__(self, local_jupyter_timeout: float, tool_prompt: str, sandbox=None):

        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox
        
        self._owns_session = sandbox is None
        
        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()

    def _ensure_session(self):

        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code: str) -> str:

        lines = code.strip().split('\n')

        if not lines:
            return code

        last_line = lines[-1].strip()

        if 'print' in last_line or 'import' in last_line:
            return code

        if not last_line:
            return code

        if last_line.startswith('#'):
            return code

        lines[-1] = 'print(' + last_line + ')'

        return '\n'.join(lines)

    @property
    def instruction(self) -> str:

        return self._tool_prompt

    @property
    def tool_config(self) -> ToolNamespaceConfig:

        return ToolNamespaceConfig(
            name='python', 
            description=self.instruction, 
            tools=[]
        )

    def _make_response(self, output: str, channel: str | None = None) -> Message:

        content = TextContent(text=output)
        author = Author(role=Role.TOOL, name='python')
        message = Message(author=author, content=[content]).with_recipient('assistant')

        if channel:
            message = message.with_channel(channel)

        return message

    def process_sync_plus(self, message: Message) -> list[Message]:

        self._ensure_session()
        raw_script = message.content[0].text
        final_script = self._ensure_last_print(raw_script)

        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)

            except TimeoutError as exc:
                output = f'[ERROR] {exc}'

        return [self._make_response(output, channel=message.channel)]

In [17]:
class AIMO3Solver:

    def __init__(self, cfg, port: int = 8000):
    
        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'
        self.template = AIMO3Template()
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()
    
        self._preload_model_weights()
        
        self.server_process = self._start_server()
    
        self.client = OpenAI(
            base_url=self.base_url, 
            api_key=self.api_key, 
            timeout=self.cfg.session_timeout
        )
    
        self._wait_for_server()
        self._initialize_kernels()
    
    def _preload_model_weights(self) -> None:
    
        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start_time = time.time()
        
        files_to_load = []
        total_size = 0
    
        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)
    
                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)
    
        def _read_file(path: str) -> None:
    
            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            list(executor.map(_read_file, files_to_load))
    
        elapsed = time.time() - start_time
        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')
    
    def _start_server(self) -> subprocess.Popen:
    
        cmd = [
            sys.executable, 
            '-m', 
            'vllm.entrypoints.openai.api_server', 
            '--seed', 
            str(self.cfg.seed), 
            '--model', 
            self.cfg.model_path, 
            '--served-model-name', 
            self.cfg.served_model_name, 
            '--tensor-parallel-size', 
            '1', 
            '--max-num-seqs', 
            str(self.cfg.batch_size), 
            '--gpu-memory-utilization', 
            str(self.cfg.gpu_memory_utilization), 
            '--host', 
            '0.0.0.0', 
            '--port', 
            str(self.port), 
            '--dtype', 
            self.cfg.dtype, 
            '--kv-cache-dtype', 
            self.cfg.kv_cache_dtype, 
            '--max-model-len', 
            str(self.cfg.context_tokens), 
            '--stream-interval', 
            str(self.cfg.stream_interval), 
            '--async-scheduling', 
            '--disable-log-stats', 
            '--enable-prefix-caching'
        ]
    
        self.log_file = open('vllm_server.log', 'w')
    
        return subprocess.Popen(
            cmd, 
            stdout=self.log_file, 
            stderr=subprocess.STDOUT, 
            start_new_session=True
        )
    
    def _wait_for_server(self):
    
        print('Waiting for vLLM server...')
        start_time = time.time()
    
        for _ in range(self.cfg.server_timeout):
            return_code = self.server_process.poll()
    
            if return_code is not None:
                self.log_file.flush()
    
                with open('vllm_server.log', 'r') as log_file:
                    logs = log_file.read()
    
                raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')
    
            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f'Server is ready (took {elapsed:.2f} seconds).\n')
    
                return
    
            except Exception:
                time.sleep(1)
    
        raise RuntimeError('Server failed to start (timeout).\n')
    
    def _initialize_kernels(self) -> None:
    
        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start_time = time.time()
    
        self.sandbox_pool = queue.Queue()
    
        def _create_sandbox():
            
            return AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(_create_sandbox) for _ in range(self.cfg.workers)]
    
            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())
    
        elapsed = time.time() - start_time
        print(f'Kernels initialized in {elapsed:.2f} seconds.\n')
    
    def _process_attempt(
        self,
        problem: str,
        system_prompt: str,
        attempt_index: int
    ) -> str:

        local_tool = None
        sandbox = None
        trace_parts = []

        attempt_seed = int(math.pow(self.cfg.seed + attempt_index, 2))

        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)

            local_tool = AIMO3Tool(
                local_jupyter_timeout=self.cfg.jupyter_timeout,
                tool_prompt=self.cfg.tool_prompt,
                sandbox=sandbox
            )

            encoding = self.encoding
            messages = self.template.apply_chat_template(
                system_prompt,
                problem,
                local_tool.tool_config
            )

            conversation = Conversation.from_messages(messages)

            for _ in range(self.cfg.turns):
                prompt_ids = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
                max_tokens = self.cfg.context_tokens - len(prompt_ids)

                if max_tokens < self.cfg.buffer_tokens:
                    break

                stream = self.client.completions.create(
                    model=self.cfg.served_model_name,
                    temperature=self.cfg.temperature,
                    logprobs=self.cfg.top_logprobs,
                    max_tokens=max_tokens,
                    prompt=prompt_ids,
                    seed=attempt_seed,
                    stream=True,
                    extra_body={
                        'min_p': self.cfg.min_p,
                        'stop_token_ids': self.stop_token_ids,
                        'return_token_ids': True
                    }
                )

                try:
                    token_buffer = []
                    text_chunks = []

                    for chunk in stream:
                        new_tokens = chunk.choices[0].token_ids
                        new_text = chunk.choices[0].text

                        if new_tokens:
                            token_buffer.extend(new_tokens)
                            text_chunks.append(new_text)

                finally:
                    stream.close()

                if not token_buffer:
                    break

                assistant_text = ''.join(text_chunks)
                trace_parts.append(f'[ASSISTANT]\n{assistant_text}\n')

                new_messages = encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT)
                conversation.messages.extend(new_messages)
                last_message = new_messages[-1]

                if last_message.channel == 'final':
                    break

                if last_message.recipient == 'python':
                    tool_responses = local_tool.process_sync_plus(last_message)
                    response_text = tool_responses[0].content[0].text
                    trace_parts.append(f'[PYTHON OUTPUT]\n{response_text}\n')
                    conversation.messages.extend(tool_responses)

        except Exception as exc:
            trace_parts.append(f'[ERROR]\n{exc}\n')

        finally:
            if sandbox is not None:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)

        return '\n'.join(trace_parts)
    
    def run_attempt(self, problem: str, problem_id: str, attempt_index: int) -> tuple:
        """Run one attempt for a problem.
        Returns (problem_id, attempt_index, answer, trace).
        This is the unit of work handed to the thread pool.
        """
        user_input = f'{problem} {self.cfg.preference_prompt}'
        trace = self._process_attempt(user_input, self.cfg.system_prompt, attempt_index)
        answer = extract_answer(trace)

        output_path = f'/kaggle/working/{int(problem_id):04d}_attempt{attempt_index + 1:02d}.txt'
        with open(output_path, 'w') as f:
            f.write(f'Problem ID : {problem_id}\n')
            f.write(f'Attempt    : {attempt_index + 1}/{self.cfg.attempts}\n')
            f.write(f'Answer     : {answer}\n\n')
            f.write(f'Problem: {problem}\n\n')
            f.write(trace)

        #print(f'  [P{problem_id} A{attempt_index + 1}] answer={answer}')
        return problem_id, attempt_index, answer, trace
    
    def __del__(self):
    
        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()
    
        if hasattr(self, 'log_file'):
            self.log_file.close()
    
        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()
    
                except Exception:
                    pass


In [18]:
solver = AIMO3Solver(CFG)

Loading model weights from /kaggle/input/models/danielhanchen/gpt-oss-120b/transformers/default/1/ into OS Page Cache...
Processed 26 files (65.28 GB) in 141.99 seconds.

Waiting for vLLM server...
Server is ready (took 121.89 seconds).

Initializing 16 persistent Jupyter kernels...
Kernels initialized in 2.79 seconds.



In [19]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    
    #id_value = id_.item(0)
    #question_text = question.item(0)

    id_value = id_
    question_text = question
    gc.disable()
    
    final_answer = solver.solve_problem(question_text, id_value)
    
    gc.enable()
    gc.collect()
    
    return pl.DataFrame({'id': id_value, 'answer': final_answer})

In [20]:
#inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

#if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
#    inference_server.serve()
    
#else:
#    inference_server.run_local_gateway(
#        ('/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv',)
#    )

In [21]:
#import json

#with open('/kaggle/input/imo-problems-slutions/IMO_Problems_Slutions.jsonl', 'r') as f:
#    for line in f:
#        data = json.loads(line)
#        year = data['year']
#        problem_num = data['problem_i']
#        question = data['Problem']
#        if int(year) <= 1983:
#            continue

#        predict(str(year) + "-" + str(problem_num), question)

In [22]:
#import json

#idx = 1
#with open('/kaggle/input/datasets/nguyennguyen599/astral-math-v1/stage2.jsonl', 'r') as f:
#    for line in f:
#        data = json.loads(line)
#        question = data['question']
#        predict(str(idx), question)
#        idx += 1

In [ ]:
import json

NUM_ATTEMPTS = CFG.attempts   # 10
MAX_WORKERS  = CFG.workers    # 16

# ── Load problems ──────────────────────────────────────────────────────────────
problems = []
with open('/kaggle/input/datasets/nguyennguyen599/astral-math-v1/stage2.jsonl', 'r') as f:
    for idx, line in enumerate(f, start=1):
        if idx < 1016:
            continue
        data = json.loads(line)
        problems.append((str(idx), data['question']))

total_tasks = len(problems) * NUM_ATTEMPTS
print(f'Problems      : {len(problems)}')
print(f'Attempts each : {NUM_ATTEMPTS}')
print(f'Total tasks   : {total_tasks}')
print(f'Workers       : {MAX_WORKERS}')
print(f'Initial fill  : {NUM_ATTEMPTS} slots → P1, {MAX_WORKERS - NUM_ATTEMPTS} slots → P2\n')

# ── Result tracking ────────────────────────────────────────────────────────────
# problem_id -> list of (attempt_index, answer)
all_results   = defaultdict(list)
results_lock  = threading.Lock()
completed_problems = set()

output_dir    = '/kaggle/working'
completed_count = 0

def write_problem_summary(problem_id: str, question: str, attempts: list) -> None:
    """Write a summary file once all attempts for a problem are done."""
    attempts_sorted = sorted(attempts, key=lambda x: x[0])   # sort by attempt index
    answers = [a for _, a in attempts_sorted]
    valid   = [a for a in answers if a is not None]
    majority = Counter(valid).most_common(1)[0][0] if valid else None

    path = f'{output_dir}/{int(problem_id):04d}_summary.txt'
    with open(path, 'w') as f:
        f.write(f'Problem ID      : {problem_id}\n')
        f.write(f'Question        : {question}\n\n')
        f.write(f'Per-attempt answers:\n')
        for aidx, ans in attempts_sorted:
            f.write(f'  Attempt {aidx + 1:02d}: {ans}\n')
        f.write(f'\nAll answers     : {answers}\n')
        f.write(f'Valid answers   : {valid}\n')
        f.write(f'Majority answer : {majority}\n')

    print(f'\n>>> Problem {problem_id} complete | question = {question[:20]}, answers={answers} | majority={majority} <<<\n')

# ── Build work list: all attempts of P1 first, then P2, etc. ──────────────────
# ThreadPoolExecutor queues tasks in submission order, so the first MAX_WORKERS
# tasks to run are exactly the first 10 (all of P1) + 6 (start of P2).
# As any thread finishes, the next queued task starts — naturally greedy.
work_items = [
    (problem_id, question, attempt_index)
    for problem_id, question in problems
    for attempt_index in range(NUM_ATTEMPTS)
]

questions = {pid: q for pid, q in problems}

# ── Execute ────────────────────────────────────────────────────────────────────
problem_futures = defaultdict(list)
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    future_map = {
        executor.submit(solver.run_attempt, question, problem_id, attempt_index):
            (problem_id, attempt_index)
        for problem_id, question, attempt_index in work_items
    }

    for future, (problem, attempt_index) in future_map.items():
        problem_futures[problem].append(future)
        
    for future in as_completed(future_map):
        problem_id, attempt_index = future_map[future]
        completed_count += 1

        try:
            pid, aidx, answer, _ = future.result()

            with results_lock:
                all_results[pid].append((aidx, answer))
                done = len(all_results[pid]) == NUM_ATTEMPTS
                valid = [a for _, a in all_results[pid] if a is not None]
                counts = Counter(valid)
                top_count = counts.most_common(1)[0][1] if counts else 0
                early_stop = top_count >= CFG.early_stop
                if (done or early_stop) and pid not in completed_problems:
                    completed_problems.add(pid)
                    snapshot = list(all_results[pid])
                else:
                    snapshot = None

            #print(f'[{completed_count}/{total_tasks}] P{pid} attempt {aidx + 1} → {answer}')

            if snapshot is not None:
                cancelled = sum(1 for f in problem_futures[pid] if not f.done() and f.cancel())
                if cancelled:
                    print(f'  → Early stop for P{pid} ({top_count}/{CFG.early_stop} agree) — cancelled {cancelled} attempts')
                write_problem_summary(pid, questions[pid], snapshot)

        except Exception as exc:
            with results_lock:
                all_results[problem_id].append((attempt_index, None))
                done = len(all_results[problem_id]) == NUM_ATTEMPTS
                if done:
                    snapshot = list(all_results[problem_id])

            print(f'[{completed_count}/{total_tasks}] P{problem_id} attempt {attempt_index + 1} FAILED: {exc}')

            if done:
                write_problem_summary(problem_id, questions[problem_id], snapshot)

print(f'\nAll {total_tasks} tasks done across {len(problems)} problems.')


Problems      : 5226
Attempts each : 5
Total tasks   : 26130
Workers       : 16
Initial fill  : 5 slots → P1, 11 slots → P2


>>> Problem 1017 complete | question = From a point on the , answers=[2, 2, 2, 4, 8] | majority=2 <<<


>>> Problem 1018 complete | question = Five, let the base o, answers=[1, 0, 1, 2, 0] | majority=1 <<<



In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for name in files:
        print(os.path.join(root, name))